# **Machine Learning — Model Development and Fairness Evaluation**

## Objectives

* Quantify the distortion introduced by the leaked `duration` feature identified in Notebook 02
* Build a deployable targeting model that excludes leakage
* Evaluate performance using metrics appropriate to an 8:1 class imbalance, not accuracy
* Measure whether the model's targeting falls evenly across age band and education level
* Produce a documented recommendation on whether the model should be deployed

## Inputs

* Data_Set/clean_data/v1/bank_marketing_cleaned.csv — cleaned dataset from Notebook 01

## Outputs

* Data_Set/models/v1/ — fitted model pipelines
* Data_Set/outputs/v1/model_comparison.csv — performance metrics
* Data_Set/outputs/v1/fairness_metrics.csv — subgroup disparity summary
* Data_Set/outputs/v1/fairness_by_group.csv — per-group fairness detail
* Data_Set/outputs/v1/figures/ — leakage, confusion matrix, feature importance and fairness charts

## Additional Comments

* Accuracy is not reported as a headline metric. At an 11% positive rate, a model predicting
  "no" for every client scores 89% accuracy while identifying nobody. ROC-AUC, recall and
  precision are used instead.

* Model A retains `duration` and exists solely to quantify what leakage does to apparent
  performance. It is never a deployment candidate. Model B excludes it and is carried forward.

* Age, education, job and marital status are excluded from the model's feature set but
  retained in the dataset, because they are required to measure whether targeting falls
  evenly across groups. This is the reasoning recorded in Notebook 01: demographics are
  analysed, not modelled.

* Fairness is measured using demographic parity difference and equal opportunity difference.
  These two definitions cannot generally be satisfied simultaneously, and the choice between
  them is stated rather than assumed.

* A false positive here means a client contacted who had no interest. For a client in a group
  more likely to be considered vulnerable, repeated unsolicited contact is a harm in itself,
  not a wasted call.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/fair-marketing-analytics/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/fair-marketing-analytics'

# Section 1

# Section 1 — Setup and Feature Preparation

In [4]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

version = 'v1'
clean_dir = f'Data_Set/clean_data/{version}'
output_dir = f'Data_Set/outputs/{version}'
model_dir = f'Data_Set/models/{version}'
fig_dir = f'{output_dir}/figures'

os.makedirs(model_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

RANDOM_STATE = 42

PALETTE = {
    'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73',
    'red': '#D55E00', 'grey': '#999999',
}

plt.rcParams.update({
    'figure.figsize': (10, 6), 'savefig.dpi': 150, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

print(f"Models to {model_dir}")
print(f"Outputs to {output_dir}")

Models to Data_Set/models/v1
Outputs to Data_Set/outputs/v1


In [5]:
df = pd.read_csv(f'{clean_dir}/bank_marketing_cleaned.csv')
print(f"{len(df):,} records, {df.shape[1]} columns")
print(f"Positive rate: {df['subscribed'].mean():.2%}")

41,176 records, 23 columns
Positive rate: 11.27%


## Feature sets

Two feature sets are defined.

**Set A** retains `duration`. Notebook 02 confirmed this field is target leakage: it is not
known when a targeting decision is made, and rank-biserial correlation with the outcome was
-0.637. Set A exists only to quantify how much apparent performance leakage buys. It is not
a deployment candidate.

**Set B** excludes `duration`. It retains campaign, macroeconomic and client-status
variables but excludes `age`, `age_band`, `job`, `education` and `marital`. Those fields
remain in the dataset for fairness measurement but are not model inputs, following the
reasoning recorded in Notebook 01. Set B is the deployment candidate.

`default_disclosed` is retained because Notebook 01 established that the original `default`
field recorded only three positive cases and functioned as a disclosure indicator. Whether a
model should reward disclosure is revisited in Section 5.

In [6]:
demographic = ['age', 'age_band', 'job', 'education', 'marital']
leakage = ['duration']

campaign = ['contact', 'month', 'day_of_week', 'campaign', 'pdays',
            'previous', 'poutcome', 'contacted_before']
macro = ['emp_var_rate', 'cons_price_idx', 'cons_conf_idx',
         'euribor3m', 'nr_employed']
client = ['housing', 'loan', 'default_disclosed']

FEATURES = {
    'A_with_leakage': campaign + macro + client + leakage,
    'B_deployable':   campaign + macro + client,
}

for name, cols in FEATURES.items():
    print(f"{name}: {len(cols)} features")

A_with_leakage: 17 features
B_deployable: 16 features


In [7]:
X_all = df.drop(columns=['subscribed'])
y = df['subscribed']

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {len(X_train_all):,}  positive rate {y_train.mean():.2%}")
print(f"Test:  {len(X_test_all):,}  positive rate {y_test.mean():.2%}")

Train: 30,882  positive rate 11.27%
Test:  10,294  positive rate 11.27%


The split is stratified so both sets retain the 11% positive rate. Without stratification a
random split could hand the test set a materially different class balance and make the
performance figures unreliable.

The full feature frame is split once and subset per model afterwards, so both models are
evaluated on exactly the same clients. This also preserves the demographic columns in the
test set for the fairness analysis in Section 4, even though they are not model inputs.

---

# Section 2

# Section 2 — Model Training

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
